# MMD GANs


## the vanilla GAN objective

$$\min_\psi \max_\theta \; \mathbb E_{x\sim P}[\log D_\theta(x)] + \mathbb E_z[\log(1 - D_\theta(G_\psi(z)))]
\iff \min_\psi \mathrm{JS}(P \,\|\, Q_\psi)$$

> If at the start of training the supports of $P$ and $Q_\psi$ do not overlap, JS is constant, and
> the generator receives no gradient.


## Integral probability metric

$$D_{\mathcal F}(P,Q) = \sup_{f \in \mathcal F} \mathbb E_{x\sim P}[f(x)] - \mathbb E_{y \sim Q}[f(y)]
\qquad (\mathcal F \text{ 1-Lipschitz} \Rightarrow \text{Wasserstein})$$

WGANs : $\mathcal F = \{f : \|f\|_L \le 1\}$.

### Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

if torch.cuda.is_available():
    device = torch.device('cuda')
    dtype = torch.cuda.FloatTensor
else:
    device = torch.device('cpu')
    dtype = torch.FloatTensor

# If you don't want to bother with the device, stay on cpu:
# device = torch.device('cpu')

print(device)

### Target Measure

In the following cell, we define the discrete target measure $\nu$ that will serve as dataset for this practical session.

The variable `xgrid` contains a grid of points that will be useful below to display the discriminators along training.

In [ ]:
d = 2   # dimension of the data points

# First dataset with 6 points
"""
n = 6
y = torch.zeros((n,d), device=device)
y[0, 0] = 0.9
y[0, 1] = 0.2
y[1, 0] = 0.75
y[1, 1] = 0.8
y[2, 0] = 0.3
y[2, 1] = 0.4
y[3, 0] = 0.4
y[3, 1] = 0.7
y[4, 0] = 0.45
y[4, 1] = 0.75
y[5, 0] = 0.7
y[5, 1] = 0.5
"""
# # Second dataset with 100 points
n = 100
t = torch.pi*torch.linspace(-.2,1.2,n)
y = .05*torch.randn((n,d))
y[:,0] += torch.cos(t)
y[:,1] += torch.sin(2*t)
y = .5 + .3*y
y = y.to(device)

# Define masses (empirical measure on the data points)
nu = torch.ones(n, device=device)/n

# generate grid for plotting purpose
nr,nc = 256,256
extent = ((-0.5/nc, 1-0.5/nc, 1-0.5/nr, -0.5/nr))
xs = torch.linspace(0, 1, steps=nr)
ys = torch.linspace(0, 1, steps=nc)
xm, ym = torch.meshgrid(xs, ys, indexing='ij')
xm = xm.T
ym = ym.T
xgrid = torch.cat((xm.reshape(nr*nc,1),ym.reshape(nr*nc,1)),1).to(device)

# Plot data points
fig = plt.figure(dpi=100)
plt.xticks([])
plt.yticks([])
plt.imshow(np.zeros((nr,nc)),cmap = 'Oranges', extent=extent) # background
plt.scatter(y[:, 0].cpu(), y[:,1].cpu(),c='navy')
plt.show()

## Define Generator architecture



In [ ]:
class Generator(torch.nn.Module):

    def __init__(self, n_in, n_out, n_hid=10, nlayers=3, device=torch.device("cpu")):
        super(Generator, self).__init__()

        self.n_in = n_in
        self.n_out = n_out
        self.n_hid = n_hid
        self.nlayers = nlayers
        self.hidden = nn.ModuleList()

        for n in range(nlayers):
            n_in_t = n_in if n==0 else n_hid
            self.hidden.append(nn.Sequential(
            nn.Linear(n_in_t, n_hid),
            nn.ELU(1)
        ).to(device))

        self.out = nn.Sequential(
            nn.Linear(n_hid, n_out),
            nn.Sigmoid()
        ).to(device)

        self.apply(self._init_weights)


    def forward(self, x):
        for n in range(self.nlayers):
            x = self.hidden[n](x)
        x = self.out(x)
        return x


    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.xavier_uniform_(module.weight, 1.0)
            if module.bias is not None:
                module.bias.data.zero_()


In [ ]:
torch.manual_seed(0)  # initialize random seed for reproducibility

n_in = 10    # dimension of the input noise
b = 100      # batch size

# Initialize generator
G = Generator(n_in=n_in, n_out=d, n_hid=100, nlayers=3, device=device)

# Draw a batch x of generated points
#    Input noise z : standard normal with shape (b, n_in)

x= torch.randn(b,n_in)

# Plot
xd = x.detach()
fig = plt.figure(dpi=100)
plt.xticks([])
plt.yticks([])
plt.imshow(np.zeros((nr,nc)),cmap = 'Oranges', extent=extent) # background
plt.scatter(xd[:, 0].cpu(), xd[:,1].cpu(),c='deepskyblue',alpha=.5)
plt.scatter(y[:, 0].cpu(), y[:,1].cpu(),c='navy')
plt.show()

## Define Discriminator Architecture

In [ ]:
class Discriminator(nn.Module):
  def __init__(self, n_in, n_hid=10):
    super(Discriminator, self).__init__()

    self.n_hid = n_hid
    self.n_in = n_in

    self.fc1 = nn.Linear(n_in, n_hid)
    self.fc2 = nn.Linear(n_hid, n_hid)
    self.fc3 = nn.Linear(n_hid, 1)

  def forward(self, x):
    y = nn.LeakyReLU(negative_slope=0.2)(self.fc1(x))
    y = nn.LeakyReLU(negative_slope=0.2)(self.fc2(y))
    y = self.fc3(y)
    return y

## Discriminator training with Weight clipping

### What `D` is estimating

With $\mathcal F$ the 1-Lipschitz functions, the IPM above is the Kantorovich–Rubinstein dual of the
1-Wasserstein distance:

$$W_1(P,Q) \;=\; \sup_{\|f\|_L \le 1} \; \mathbb E_{x\sim P}[f(x)] - \mathbb E_{y\sim Q}[f(y)]$$

The critic loss minimised below, $\mathbb E_z[D(G(z))] - \mathbb E_{x \sim P}[D(x)]$, is therefore
the negative of that dual objective, and the stored `-Dloss` is a **lower** bound on
$W_1(P, Q_\psi)$ — a bound rather than the value, because a finite network only approximates the
supremum, and because the bound is meaningful only while $D$ stays inside $\mathcal F$.

**Weight clipping** enforces $\mathcal F$ the blunt way: after every step each parameter is
projected back into $[-c,c]$, which bounds each layer's norm and hence the Lipschitz constant of the
composition. It constrains more than required — it also caps what $D$ can express, which is what the
gradient penalty below is designed to avoid.

## Estimation of the Lipschitz constant of the discriminator

The following function computes a lower bound of the Lipschitz constant of $D$ on points that are interpolated between $x$ and $y$.

NB: If $x$ and $y$ do not have the same number of points, we discard the last points.
In comment, we give an alternative code that allows to compute all segments $[x_i, y_j]$ (but it is, of course, slower that just computing segments $[x_i, y_i]$).

### Reading the constraint off the gradient

For a differentiable $D$,

$$\|D\|_L \;=\; \sup_{u \neq v} \frac{|D(u) - D(v)|}{\|u - v\|} \;=\; \sup_x \|\nabla D(x)\|,$$

so evaluating $\|\nabla D\|$ anywhere gives a lower bound. The cell below evaluates it on random
interpolates $\hat x = \alpha y_i + (1-\alpha) x_i$, $\alpha \sim \mathcal U[0,1]$: the segments
between real and generated points are where the optimal dual witness has its unit-slope region, so
that is where the constraint is tight and the bound is informative.

In [ ]:
def lipconstant(D,x,y):
    b = x.shape[0]
    n = y.shape[0]
    # shrink vectors if they are too large
    if n>b:
        y = y[0:b,:]
        n = b
    else:
        x = x[0:n,:]
        b = n
    # compute interpolated points
    alpha = torch.rand((b,1),device=device)
    interp = alpha * y + (1 - alpha) * x
    interp.requires_grad_()

    # Calculate probability of interpolated examples
    Di = D(interp).view(-1)

    # Calculate gradients of probabilities with respect to examples
    gradout = torch.ones(Di.size()).to(device)
    gradients = torch.autograd.grad(outputs=Di, inputs=interp, grad_outputs=gradout,
       create_graph=True, retain_graph=True)[0]

    # Derivatives of the gradient close to 0 can cause problems because of
    # the square root, so manually calculate norm and add epsilon
    gradients_norm = torch.sqrt(torch.sum(gradients ** 2, dim=1))

    # Return gradient penalty
    return torch.mean(gradients_norm)

#     # Calculate interpolation
#     b = x.shape[0]
#     n = y.shape[0]
#     alpha = torch.rand((b,n,1),device=device)
#     interp = (alpha * y[None,:,:] + (1 - alpha) * x[:,None,:]).flatten(end_dim=1)
#     interp.requires_grad_()

In [ ]:
lip_cst =lipconstant(D,xd, y)
print(f"Lower bound of the lipshitz constant : {lip_cst}")

## Gradient Penalty


$$ GP(D) = \mathbb{E}\left[ \Big(\|\nabla D(X)\| - 1 \Big)^2 \right] .$$
In pratice :
$$\|\nabla D(X)\| \approx \sqrt{\|\nabla D (X)\|_2^2 + 10^{-12} } $$


Added to the critic objective, this gives the WGAN-GP problem solved in the next cells:

$$\mathcal L_D \;=\;
\underbrace{\mathbb E_z\big[D(G(z))\big] - \mathbb E_{x\sim P}\big[D(x)\big]}_{-\;\text{dual objective}}
\;+\; \lambda\,\underbrace{\mathbb E_{\hat x}\Big[\big(\|\nabla D(\hat x)\| - 1\big)^2\Big]}_{GP(D)},
\qquad \hat x = \alpha x + (1-\alpha)\,G(z).$$

Unlike clipping this is a *soft* constraint,

In [ ]:
def gradient_penalty(D,x,y):
    b = x.shape[0]
    n = y.shape[0]
    # shrink vectors if they are too large
    if n>b:
        y = y[0:b,:]
        n = b
    else:
        x = x[0:n,:]
        b = n
    # compute interpolated points
    alpha = torch.rand((b,1),device=device)
    interp = (alpha * y + (1 - alpha) * x)
    interp.requires_grad_()

#     # Calculate interpolation
#     b = x.shape[0]
#     n = y.shape[0]
#     alpha = torch.rand((b,n,1),device=device)
#     interp = (alpha * y[None,:,:] + (1 - alpha) * x[:,None,:]).flatten(end_dim=1)
#     interp.requires_grad_()

    # Calculate probability of interpolated examples
    Di = D(interp).view(-1)

    # Calculate gradients of probabilities with respect to examples
    gradout = torch.ones(Di.size()).to(device)
    gradients = torch.autograd.grad(outputs=Di, inputs=interp, grad_outputs=gradout,
       create_graph=True, retain_graph=True)[0]

    # Derivatives of the gradient close to 0 can cause problems because of
    # the square root, so manually calculate norm and add epsilon
    epsilon=10e-12
    gradients_norm = torch.sqrt(torch.sum(gradients**2, dim=1)+epsilon)

    # Return gradient penalty
    return ((gradients_norm - 1) ** 2).mean()
#     return ((gradients_norm - 1) ** 2*(gradients_norm>1)).mean()

print(gradient_penalty(D,xd,y))

## Discriminator training with Gradient Penalty

In [ ]:
torch.manual_seed(1)  # initialize random seed for reproducibility

# Initialize generators and discriminators
G = Generator(n_in=n_in, n_out=d, n_hid=10, nlayers=3, device=device)
D = Discriminator(n_in=d, n_hid=10).to(device)

# parameters for training
lrdisc = 0.002
beta_1 = 0.5
beta_2 = 0.999
niterD=1000
gpw = 10       # parameter for gradient penalty

optimD = optim.Adam(D.parameters(), lr=lrdisc, betas=(beta_1, beta_2))


iter_display = 100  # display current configuration each iter_display iteration

# Lists to keep track of progress
Dlosses = []

for iter in range(0,niterD):
    optimD.zero_grad()
    ### UPDATE OF D ###
    z= torch.randn(b, n_in, device=device)

    # Detach fake samples so we don't backpropagate through G
    fake_samples = G(z).detach()

    # Dloss for WGAN-GP: E[D(fake)] - E[D(real)] + lambda * GP
    # y represents our real data points
    Dlosst = torch.mean(D(fake_samples)) - torch.mean(D(y)) + gpw * gradient_penalty(D, y, fake_samples)
    Dlosst.backward()

    ### SAVE LOSS ###
    optimD.step()

    Dlosses.append(-Dlosst.item())

    if(iter%iter_display == 0):
        #print('[%d/%d], Dloss=%.4f' % (iter, niterD, Dlosst.item()))
        Dxgrid = D(xgrid).detach().cpu().numpy().reshape(nr,nc)
        x = G(z)
        xd = x.detach().squeeze(1)
        strtitle = 'Iter '+str(iter)
        fig = plt.figure(dpi=100)
        plt.xticks([])
        plt.yticks([])
        plt.imshow(Dxgrid,cmap = 'Oranges', extent=extent)  # discriminator
        plt.scatter(xd[:, 0].cpu(), xd[:,1].cpu(),c='deepskyblue',alpha=.5)
        plt.scatter(y[:, 0].cpu(), y[:,1].cpu(),c='navy')
        plt.title(strtitle)
        plt.show()

print('Final estimated Lipschitz constant = ',lipconstant(D, x.detach(), y).item())

plt.figure(dpi=100)
plt.plot(Dlosses)
plt.title('Discriminator loss')
plt.show()

---
## RKHS ball


Taking $\mathcal F$ to be the unit ball of an RKHS removes the inner problem entirely:

> Take $\mathcal F$ = unit ball of an RKHS $\mathcal H$ with kernel $k$. Then the IPM collapses to a
> distance between two points: $\mathrm{MMD}(P,Q) = \|\mu_P - \mu_Q\|_{\mathcal H}$, where
> $\mu_P = \mathbb E_{X \sim P}[k(\cdot, X)]$.

$$\mathrm{MMD}^2(P,Q;\mathcal H_k) = \mathbb E_{X,X'}[k(X,X')] + \mathbb E_{Y,Y'}[k(Y,Y')]
- 2\,\mathbb E_{X,Y}[k(X,Y)]$$

$k(x,y)$ is generally chosen as a simple decreasing function of $\|x-y\|^2$:

$$\text{RBF: } k(x,y) = e^{-\frac{\|x-y\|^2}{2\sigma^2}}
\qquad\qquad \text{RQ: } k(x,y) = \Big(1 + \frac{\|x-y\|^2}{2\alpha \ell}\Big)^{-\alpha}$$

No supremum, no Lipschitz constraint, no `niterD` — three kernel sums in closed form. The optimal
witness is even available explicitly,
$f^{*}(x) \propto \mathbb E_P\, k(X,x) - \mathbb E_Q\, k(Y,x)$, which is the MMD analogue of the $D$
trained above, obtained without training anything.




In [ ]:
def rbf(X, Y, sigma=2.0):
    """
    Computes the RBF (Gaussian) kernel matrix between two sets of points.
    K(x, y) = exp(- ||x - y||^2 / (2 * sigma^2))
    """
    X_size = X.size(0)
    Y_size = Y.size(0)

    X_ = X.unsqueeze(1)  # (X_size, 1, dim)
    Y_ = Y.unsqueeze(0)  # (1, Y_size, dim)

    # Calculate squared Euclidean distance: ||X_i - Y_j||^2
    # Resulting shape: (X_size, Y_size)
    dist_sq = (X_ - Y_).pow(2).sum(2)

    # Apply the RBF kernel formula
    return torch.exp(-dist_sq / (2 * sigma**2))

In [ ]:
def MMD_from_k(k_func, X, Y, sigma=2.0):
    m = X.size(0)
    n = Y.size(0)

    K_XX = k_func(X, X, sigma) # m x m matrix of kernel evaluations between X and X
    K_YY = k_func(Y, Y, sigma) # n x n matrix of kernel evaluations between Y and Y
    K_XY = k_func(X, Y, sigma) # m x n matrix of kernel evaluations between X and Y

    # MMD^2 calculation using the biased estimator.
    # This directly corresponds to the empirical estimate of E[k(X,X')] + E[k(Y,Y')] - 2E[k(X,Y)],
    # which is derived from the optimal discriminator f*(x) = E[k(X,x)] - E[k(Y,x)] in its empirical form.
    MMD2 = K_XX.mean() + K_YY.mean() - 2 * K_XY.mean()

    # Use torch.relu to handle potential negative MMD2 due to numerical precision
    # and then take the square root to get MMD.
    return torch.sqrt(torch.relu(MMD2))

In [ ]:
def optimal_MMD_descriminator(x,X,Y,k_func,sigma):
     opt_desc= k_func(x,X,sigma).mean(dim=1)-k_func(x,Y,sigma).mean(dim=1)
     return opt_desc

In [ ]:
## parameters for training
n_epochs = 300
niterD=1000
niterG=10
gpw = .1
k= rbf
lr = 0.002      # learning rate for generator
lrdisc = 0.002  # learning rate for discriminator
beta_1 = 0.5
beta_2 = 0.999
sigma=2


torch.manual_seed(1)  # initialize random seed for reproducibility

# Initialize generators and discriminators
G = Generator(n_in=n_in, n_out=d, n_hid=10, nlayers=3, device=device)
optimG = optim.Adam(G.parameters(), lr=lr)

D = Discriminator(n_in=d, n_hid=10).to(device)
optimD = optim.Adam(D.parameters(), lr=lrdisc, betas=(beta_1, beta_2))


Glosses = []
Dlosses = []

iter_display = 5

# Main loop
for epoch in range(1,n_epochs):

    ############################


    ############################
    ### Train generator (niterG iterations)
    ############################
    for iter in range(0,niterG):
        optimG.zero_grad()
        z = torch.randn(b, n_in, device=device)
        fake_samples = G(z)
        Gloss = MMD_from_k(k,fake_samples,y, sigma=sigma)
        Gloss.backward()
        optimG.step()
        # define x for later stats/plotting

    # Output training stats
    # Replaced meanvariation with lipconstant as used in earlier cells, or we can just print the losses
    print('[%d/%d] 	Loss_G: %.4f'
      % (epoch, n_epochs, Gloss.item()))
    Glosses.append(Gloss.item())


    if(epoch % iter_display == 0):
        Dxgrid = optimal_MMD_descriminator(xgrid,y,fake_samples,rbf,sigma).detach().cpu().numpy().reshape(nr,nc)
        z = torch.randn(b, n_in, device=device)
        x = G(z)
        xd = x.detach().squeeze(1)
        strtitle = 'Epoch '+str(epoch)
        fig = plt.figure(dpi=100)
        plt.xticks([])
        plt.yticks([])
        plt.imshow(Dxgrid,cmap = 'Oranges', extent=extent)
        plt.scatter(xd[:, 0].cpu(), xd[:,1].cpu(),c='deepskyblue',alpha=.5)
        plt.scatter(y[:, 0].cpu(), y[:,1].cpu(),c='navy')
        plt.title(strtitle)
        plt.show()


plt.figure(dpi=100)
plt.plot(Glosses)
plt.title('Generator loss')
plt.show()

# Save final generator for later use
wgan = Generator(n_in=n_in, n_out=d, n_hid=10, nlayers=3, device=device)
wgan.load_state_dict(G.state_dict())

### Visualization of Fake vs. Real Samples

In [ ]:
fig = plt.figure(dpi=100)
plt.xticks([])
plt.yticks([])
plt.imshow(D(xgrid).detach().cpu().numpy().reshape(nr,nc), cmap='Oranges', extent=extent) # Discriminator's output as background
plt.scatter(x[:, 0].detach().cpu(), x[:,1].detach().cpu(), c='deepskyblue', alpha=.5, label='Fake Samples')
plt.scatter(y[:, 0].cpu(), y[:,1].cpu(), c='navy', label='Real Samples')
plt.title('Current Distribution: Fake vs. Real Samples')
plt.legend()
plt.show()

## Comment on the learning process with $\sigma =2$
The quality of the training is poor, the generator failes to genrate points close to the original distribution. **This is because of the high value of $\sigma$**: the MMD loss takes a value close to $0$ values no matter if the generated ponts are close to the distribution of the original points or not.

#Kernel hyperparameters tuning

**$\sigma=0.5$**

In [ ]:
## parameters for training
n_epochs = 300
niterD=1000
niterG=10
gpw = .1
k= rbf
lr = 0.002      # learning rate for generator
  # learning rate for discriminator
beta_1 = 0.5
beta_2 = 0.999
sigma=0.5


torch.manual_seed(1)  # initialize random seed for reproducibility

# Initialize generators and discriminators
G = Generator(n_in=n_in, n_out=d, n_hid=10, nlayers=3, device=device)
optimG = optim.Adam(G.parameters(), lr=lr)

D = Discriminator(n_in=d, n_hid=10).to(device)
optimD = optim.Adam(D.parameters(), lr=lrdisc, betas=(beta_1, beta_2))


Glosses = []
Dlosses = []

iter_display = 5

# Main loop
for epoch in range(1,n_epochs):

    ############################


    ############################
    ### Train generator (niterG iterations)
    ############################
    for iter in range(0,niterG):
        optimG.zero_grad()
        z = torch.randn(b, n_in, device=device)
        fake_samples = G(z)
        Gloss = MMD_from_k(k,fake_samples,y, sigma=sigma)
        Gloss.backward()
        optimG.step()
        # define x for later stats/plotting

    # Output training stats
    # Replaced meanvariation with lipconstant as used in earlier cells, or we can just print the losses
    print('[%d/%d] 	Loss_G: %.4f'
      % (epoch, n_epochs, Gloss.item()))
    Glosses.append(Gloss.item())


    if(epoch % iter_display == 0):
        Dxgrid = optimal_MMD_descriminator(xgrid,y,fake_samples,rbf,sigma).detach().cpu().numpy().reshape(nr,nc)
        z = torch.randn(b, n_in, device=device)
        x = G(z)
        xd = x.detach().squeeze(1)
        strtitle = 'Epoch '+str(epoch)
        fig = plt.figure(dpi=100)
        plt.xticks([])
        plt.yticks([])
        plt.imshow(Dxgrid,cmap = 'Oranges', extent=extent)
        plt.scatter(xd[:, 0].cpu(), xd[:,1].cpu(),c='deepskyblue',alpha=.5)
        plt.scatter(y[:, 0].cpu(), y[:,1].cpu(),c='navy')
        plt.title(strtitle)
        plt.show()


plt.figure(dpi=100)
plt.plot(Glosses)
plt.title('Generator loss')
plt.show()

# Save final generator for later use
wgan = Generator(n_in=n_in, n_out=d, n_hid=10, nlayers=3, device=device)
wgan.load_state_dict(G.state_dict())

**$\sigma = 0.1$**

In [ ]:
## parameters for training
n_epochs = 300
niterD=1000
niterG=10
gpw = .1
k= rbf
lr = 0.002      # learning rate for generator
  # learning rate for discriminator
beta_1 = 0.5
beta_2 = 0.999
sigma=0.5


torch.manual_seed(1)  # initialize random seed for reproducibility

# Initialize generators and discriminators
G = Generator(n_in=n_in, n_out=d, n_hid=10, nlayers=3, device=device)
optimG = optim.Adam(G.parameters(), lr=lr)

D = Discriminator(n_in=d, n_hid=10).to(device)
optimD = optim.Adam(D.parameters(), lr=lrdisc, betas=(beta_1, beta_2))


Glosses = []
Dlosses = []

iter_display = 5

# Main loop
for epoch in range(1,n_epochs):

    ############################


    ############################
    ### Train generator (niterG iterations)
    ############################
    for iter in range(0,niterG):
        optimG.zero_grad()
        z = torch.randn(b, n_in, device=device)
        fake_samples = G(z)
        Gloss = MMD_from_k(k,fake_samples,y, sigma=sigma)
        Gloss.backward()
        optimG.step()
        # define x for later stats/plotting

    # Output training stats
    # Replaced meanvariation with lipconstant as used in earlier cells, or we can just print the losses
    print('[%d/%d] 	Loss_G: %.4f'
      % (epoch, n_epochs, Gloss.item()))
    Glosses.append(Gloss.item())


    if(epoch % iter_display == 0):
        Dxgrid = optimal_MMD_descriminator(xgrid,y,fake_samples,rbf,sigma).detach().cpu().numpy().reshape(nr,nc)
        z = torch.randn(b, n_in, device=device)
        x = G(z)
        xd = x.detach().squeeze(1)
        strtitle = 'Epoch '+str(epoch)
        fig = plt.figure(dpi=100)
        plt.xticks([])
        plt.yticks([])
        plt.imshow(Dxgrid,cmap = 'Oranges', extent=extent)
        plt.scatter(xd[:, 0].cpu(), xd[:,1].cpu(),c='deepskyblue',alpha=.5)
        plt.scatter(y[:, 0].cpu(), y[:,1].cpu(),c='navy')
        plt.title(strtitle)
        plt.show()


plt.figure(dpi=100)
plt.plot(Glosses)
plt.title('Generator loss')
plt.show()

# Save final generator for later use
wgan = Generator(n_in=n_in, n_out=d, n_hid=10, nlayers=3, device=device)
wgan.load_state_dict(G.state_dict())

**$σ=0.05$**

In [ ]:
## parameters for training
n_epochs = 300
niterD=1000
niterG=10
gpw = .1
k= rbf
lr = 0.002      # learning rate for generator
  # learning rate for discriminator
beta_1 = 0.5
beta_2 = 0.999
sigma=0.05


torch.manual_seed(1)  # initialize random seed for reproducibility

# Initialize generators and discriminators
G = Generator(n_in=n_in, n_out=d, n_hid=10, nlayers=3, device=device)
optimG = optim.Adam(G.parameters(), lr=lr)

D = Discriminator(n_in=d, n_hid=10).to(device)
optimD = optim.Adam(D.parameters(), lr=lrdisc, betas=(beta_1, beta_2))


Glosses = []
Dlosses = []

iter_display = 5

# Main loop
for epoch in range(1,n_epochs):

    ############################


    ############################
    ### Train generator (niterG iterations)
    ############################
    for iter in range(0,niterG):
        optimG.zero_grad()
        z = torch.randn(b, n_in, device=device)
        fake_samples = G(z)
        Gloss = MMD_from_k(k,fake_samples,y, sigma=sigma)
        Gloss.backward()
        optimG.step()
        # define x for later stats/plotting

    # Output training stats
    # Replaced meanvariation with lipconstant as used in earlier cells, or we can just print the losses
    print('[%d/%d] 	Loss_G: %.4f'
      % (epoch, n_epochs, Gloss.item()))
    Glosses.append(Gloss.item())


    if(epoch % iter_display == 0):
        Dxgrid = optimal_MMD_descriminator(xgrid,y,fake_samples,rbf,sigma).detach().cpu().numpy().reshape(nr,nc)
        z = torch.randn(b, n_in, device=device)
        x = G(z)
        xd = x.detach().squeeze(1)
        strtitle = 'Epoch '+str(epoch)
        fig = plt.figure(dpi=100)
        plt.xticks([])
        plt.yticks([])
        plt.imshow(Dxgrid,cmap = 'Oranges', extent=extent)
        plt.scatter(xd[:, 0].cpu(), xd[:,1].cpu(),c='deepskyblue',alpha=.5)
        plt.scatter(y[:, 0].cpu(), y[:,1].cpu(),c='navy')
        plt.title(strtitle)
        plt.show()


plt.figure(dpi=100)
plt.plot(Glosses)
plt.title('Generator loss')
plt.show()

# Save final generator for later use
wgan = Generator(n_in=n_in, n_out=d, n_hid=10, nlayers=3, device=device)
wgan.load_state_dict(G.state_dict())

**$\sigma=0.005$**

In [ ]:
## parameters for training
n_epochs = 300
niterD=1000
niterG=10
gpw = .1
k= rbf
lr = 0.002      # learning rate for generator
  # learning rate for discriminator
beta_1 = 0.5
beta_2 = 0.999
sigma=0.005


torch.manual_seed(1)  # initialize random seed for reproducibility

# Initialize generators and discriminators
G = Generator(n_in=n_in, n_out=d, n_hid=10, nlayers=3, device=device)
optimG = optim.Adam(G.parameters(), lr=lr)

D = Discriminator(n_in=d, n_hid=10).to(device)
optimD = optim.Adam(D.parameters(), lr=lrdisc, betas=(beta_1, beta_2))


Glosses = []
Dlosses = []

iter_display = 5

# Main loop
for epoch in range(1,n_epochs):

    ############################


    ############################
    ### Train generator (niterG iterations)
    ############################
    for iter in range(0,niterG):
        optimG.zero_grad()
        z = torch.randn(b, n_in, device=device)
        fake_samples = G(z)
        Gloss = MMD_from_k(k,fake_samples,y, sigma=sigma)
        Gloss.backward()
        optimG.step()
        # define x for later stats/plotting

    # Output training stats
    # Replaced meanvariation with lipconstant as used in earlier cells, or we can just print the losses
    print('[%d/%d] 	Loss_G: %.4f'
      % (epoch, n_epochs, Gloss.item()))
    Glosses.append(Gloss.item())


    if(epoch % iter_display == 0):
        Dxgrid = optimal_MMD_descriminator(xgrid,y,fake_samples,rbf,sigma).detach().cpu().numpy().reshape(nr,nc)
        z = torch.randn(b, n_in, device=device)
        x = G(z)
        xd = x.detach().squeeze(1)
        strtitle = 'Epoch '+str(epoch)
        fig = plt.figure(dpi=100)
        plt.xticks([])
        plt.yticks([])
        plt.imshow(Dxgrid,cmap = 'Oranges', extent=extent)
        plt.scatter(xd[:, 0].cpu(), xd[:,1].cpu(),c='deepskyblue',alpha=.5)
        plt.scatter(y[:, 0].cpu(), y[:,1].cpu(),c='navy')
        plt.title(strtitle)
        plt.show()


plt.figure(dpi=100)
plt.plot(Glosses)
plt.title('Generator loss')
plt.show()

# Save final generator for later use
wgan = Generator(n_in=n_in, n_out=d, n_hid=10, nlayers=3, device=device)
wgan.load_state_dict(G.state_dict())

---
## The trained generator as a prior
### restoration, conditional generation, a navigable latent space*

### Restoration without retraining

A variational restoration minimises $E(x) = f(y,x) + \lambda R(x)$. If the clean image is a
plausible output of a frozen $G_\theta$, **the prior is already learned** — we only have to search
in $z$:

$$\hat z = \arg\min_z \; f\big(y,\, G_\theta(z)\big) + \tfrac{\lambda}{2}\|z\|^2,
\qquad \hat x = G_\theta(\hat z).$$

Denoising on MNIST by gradient descent in $z$ only.


In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import math, os, time, urllib.request
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Budget preset ─────────────────────────────────────────────────────────────
PRESET = os.environ.get('MMD_PRESET', 'auto')
if PRESET == 'auto':
    PRESET = 'gpu' if torch.cuda.is_available() else 'cpu'

BUDGETS = {
    #            Part 3 ablation   Part 2 generator            critic / decoder width
    'gpu':   dict(abl_iters=3000, main_iters=10000, n_critic=5, batch=128,
                  ndf=32, ngf=64, lenet_epochs=4, enc_iters=8000, z_steps=400),
    'cpu':   dict(abl_iters=400,  main_iters=2500,  n_critic=2, batch=128,
                  ndf=16, ngf=32, lenet_epochs=2, enc_iters=1000, z_steps=300),
    'smoke': dict(abl_iters=12,   main_iters=20,    n_critic=1, batch=64,
                  ndf=16, ngf=32, lenet_epochs=1, enc_iters=12,  z_steps=15),
}
B = BUDGETS[PRESET]

torch.manual_seed(0)
np.random.seed(0)

print(f'device = {device}   preset = {PRESET!r}')
print(f'torch {torch.__version__}')
print({k: v for k, v in B.items()})

# ── Figure style for Parts 2-3 (Part 1's figures are already rendered) ───────────────────────
INK, INK2, GRIDC = '#0b0b0b', '#52514e', '#d9d8d4'
C1, C2, C3 = '#2a78d6', '#eb6834', '#1baf7a'     # categorical slots, fixed order
_BLUE_RAMP = ['#86b6ef', '#5598e7', '#2a78d6', '#1c5cab', '#104281', '#0d366b']
# signed fields (the witness) get a diverging map with a neutral midpoint
CMAP_DIV = LinearSegmentedColormap.from_list('div', ['#2a78d6', '#f0efec', '#e34948'])
CMAP_SEQ = LinearSegmentedColormap.from_list('seq', ['#fcfcfb'] + _BLUE_RAMP)

def seq(n):
    """n colours from the single-hue ordinal ramp, light -> dark (ordered variables)."""
    idx = np.linspace(0, len(_BLUE_RAMP) - 1, n)
    return [_BLUE_RAMP[int(round(i))] for i in idx]

mpl.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 120, 'figure.facecolor': 'white',
    'axes.facecolor': 'white', 'axes.edgecolor': GRIDC, 'axes.labelcolor': INK2,
    'axes.titlecolor': INK, 'axes.titlesize': 10, 'axes.titleweight': 'bold',
    'axes.labelsize': 9, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRIDC, 'grid.linewidth': 0.6, 'axes.axisbelow': True,
    'xtick.color': INK2, 'ytick.color': INK2, 'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'lines.linewidth': 2.0, 'lines.markersize': 6,
    'legend.frameon': False, 'legend.fontsize': 8, 'legend.labelcolor': INK2,
    'font.size': 9,
})

In [ ]:
# ── Kernels and the MMD of the bridge above (used by Parts 2 and 3) ───────────
def sqdist(X, Y):
    """Pairwise squared Euclidean distances, (m,d) x (n,d) -> (m,n)."""
    x2 = X.pow(2).sum(1, keepdim=True)
    y2 = Y.pow(2).sum(1, keepdim=True).t()
    return (x2 + y2 - 2.0 * (X @ Y.t())).clamp_min(0.0)

def rbf(X, Y, sigma=1.0):
    r"""k(x,y) = exp(-||x-y||^2 / (2 sigma^2))  — the deck's RBF."""
    return torch.exp(-sqdist(X, Y) / (2.0 * sigma ** 2))

def rq(X, Y, alpha=1.0, ell=1.0):
    r"""k(x,y) = (1 + ||x-y||^2 / (2 alpha l))^(-alpha)  — the deck's rational quadratic."""
    return (1.0 + sqdist(X, Y) / (2.0 * alpha * ell)).pow(-alpha)

def _mean_offdiag(M):
    n = M.size(0)
    return (M.sum() - M.diagonal().sum()) / (n * (n - 1))

def _median_base(d2):
    """Median off-diagonal squared distance: the median-heuristic bandwidth base (2 sigma^2)."""
    n = d2.size(0)
    mask = ~torch.eye(n, dtype=torch.bool, device=d2.device)
    return d2[mask].median().detach().clamp_min(1e-8)

def mmd2_biased(k, X, Y, **kw):
    """V-statistic: the plug-in estimator, diagonals included. Always >= 0."""
    return k(X, X, **kw).mean() + k(Y, Y, **kw).mean() - 2.0 * k(X, Y, **kw).mean()

def mmd2_unbiased(k, X, Y, **kw):
    """U-statistic: diagonals removed, so E[.] = MMD^2 exactly (may be < 0)."""
    return (_mean_offdiag(k(X, X, **kw)) + _mean_offdiag(k(Y, Y, **kw))
            - 2.0 * k(X, Y, **kw).mean())


In [ ]:
# ── MNIST, without a torchvision dependency ──────────────────────────────────
MNIST_URL = 'https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz'
MNIST_PATHS = ['mnist.npz', os.path.expanduser('~/.keras/datasets/mnist.npz'),
               '/root/.keras/datasets/mnist.npz']

def load_mnist():
    path = next((p for p in MNIST_PATHS if os.path.exists(p)), None)
    if path is None:
        path = 'mnist.npz'
        print('downloading MNIST ...')
        urllib.request.urlretrieve(MNIST_URL, path)
    with np.load(path) as f:
        xtr, ytr, xte, yte = f['x_train'], f['y_train'], f['x_test'], f['y_test']
    img = lambda a: torch.from_numpy(a).float().div_(127.5).sub_(1.0).unsqueeze(1)  # -> [-1,1]
    return img(xtr), torch.from_numpy(ytr).long(), img(xte), torch.from_numpy(yte).long()

x_train, y_train, x_test, y_test = load_mnist()
print(f'train {tuple(x_train.shape)}   test {tuple(x_test.shape)}   '
      f'range [{x_train.min():.0f}, {x_train.max():.0f}]')

def batches(x, y=None, bs=128, seed=0):
    """Endless shuffled minibatches straight out of memory."""
    g = torch.Generator().manual_seed(seed)
    while True:
        idx = torch.randperm(x.size(0), generator=g)
        for i in range(0, x.size(0) - bs + 1, bs):
            j = idx[i:i + bs]
            yield x[j].to(device), (None if y is None else y[j].to(device))

def make_grid(imgs, nrow=8, pad=2):
    imgs = imgs.detach().cpu().clamp(-1, 1).add(1).div(2)
    b, c, hh, ww = imgs.shape
    rows = math.ceil(b / nrow)
    canvas = torch.ones(c, rows * (hh + pad) + pad, nrow * (ww + pad) + pad)
    for i in range(b):
        r, q = divmod(i, nrow)
        canvas[:, pad + r * (hh + pad): pad + r * (hh + pad) + hh,
                  pad + q * (ww + pad): pad + q * (ww + pad) + ww] = imgs[i]
    return canvas

def show_grid(imgs, title='', nrow=8, ax=None, figsize=(4.2, 4.2)):
    g = make_grid(imgs, nrow)
    if ax is None:
        _, ax = plt.subplots(figsize=figsize)
    ax.imshow(g[0] if g.shape[0] == 1 else g.permute(1, 2, 0), cmap='gray', vmin=0, vmax=1)
    ax.set_title(title); ax.axis('off'); ax.grid(False)
    return ax

show_grid(next(batches(x_train, bs=64, seed=0))[0][:64], 'MNIST — the target')
plt.show()

In [ ]:
# ── Generator, autoencoder critic, and MMD on the codes ──────────────────────
from torch.nn.utils import spectral_norm

Z_DIM = 100

class DCGenerator(nn.Module):
    """DCGAN generator: z -> 28x28 image in [-1,1].  (Named DCGenerator so that
    Part 1's 2-D `Generator` stays intact.)"""
    def __init__(self, z_dim=Z_DIM, ngf=64, channels=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim, ngf * 4, 7, 1, 0, bias=False), nn.BatchNorm2d(ngf * 4), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False), nn.BatchNorm2d(ngf * 2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False), nn.BatchNorm2d(ngf), nn.ReLU(True),
            nn.Conv2d(ngf, channels, 3, 1, 1, bias=False), nn.Tanh())

    def forward(self, z):
        return self.net(z.view(z.size(0), -1, 1, 1) if z.dim() == 2 else z)

class AECritic(nn.Module):
    """Autoencoder critic. encode(x) -> the d-dimensional code MMD is computed on.

    The reconstruction head is what stops the encoder collapsing onto a statistic the
    generator can satisfy with blank images; spectral norm keeps the witness Lipschitz
    without clamping the feature scale.
    """
    def __init__(self, channels=1, ndf=32, ngf=64, code_dim=32):
        super().__init__()
        self.encoder = nn.Sequential(
            spectral_norm(nn.Conv2d(channels, ndf, 4, 2, 1)), nn.LeakyReLU(0.2, True),
            spectral_norm(nn.Conv2d(ndf, ndf * 2, 4, 2, 1)), nn.LeakyReLU(0.2, True),
            spectral_norm(nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1)), nn.LeakyReLU(0.2, True),
            nn.Flatten(), spectral_norm(nn.Linear(ndf * 4 * 3 * 3, code_dim)))
        self.dec_fc = nn.Sequential(nn.Linear(code_dim, ngf * 4 * 7 * 7),
                                    nn.BatchNorm1d(ngf * 4 * 7 * 7), nn.ReLU(True))
        self.ngf4 = ngf * 4
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False), nn.BatchNorm2d(ngf * 2), nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False), nn.BatchNorm2d(ngf), nn.ReLU(True),
            nn.Conv2d(ngf, channels, 3, 1, 1, bias=False), nn.Tanh())

    def encode(self, x):
        return self.encoder(x)

    def decode(self, c):
        return self.decoder(self.dec_fc(c).view(-1, self.ngf4, 7, 7))

    def forward(self, x):
        c = self.encode(x)
        return c, self.decode(c)

# RBF mixture with a median-heuristic base, as in the reference implementation:
# the bandwidth tracks the code scale instead of being hand-set.
RBF_MULTS = (0.25, 0.5, 1.0, 2.0, 4.0)
ALPHAS    = (0.2, 0.5, 1.0, 2.0, 5.0)
EPS = 1e-6

def mmd2_codes(fx, fy):
    """Unbiased MMD^2 on the critic codes, RBF mixture, median-heuristic base."""
    pooled = torch.cat([fx, fy])
    base = _median_base(sqdist(pooled, pooled))
    mix = lambda d2: sum(torch.exp(-d2 / (c * base)) for c in RBF_MULTS)
    return (_mean_offdiag(mix(sqdist(fx, fx))) + _mean_offdiag(mix(sqdist(fy, fy)))
            - 2.0 * mix(sqdist(fx, fy)).mean())

def mmd_codes(fx, fy):
    """sqrt of the clamped MMD^2 — the reference's better-scaled objective."""
    return torch.sqrt(torch.relu(mmd2_codes(fx, fy)) + EPS)

In [ ]:
# ── A frozen LeNet, for a metric the critic cannot game ──────────────────────
class LeNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(128 * 7 * 7, 256), nn.ReLU())
        self.classifier = nn.Linear(256, num_classes)

    def embed(self, x):
        return self.fc(self.features(x))      # 256-d penultimate features

    def forward(self, x):
        return self.classifier(self.embed(x))

LENET_PATH = 'checkpoints/lenet_mnist.pt'
os.makedirs('checkpoints', exist_ok=True)
lenet = LeNet().to(device)
if os.path.exists(LENET_PATH):
    lenet.load_state_dict(torch.load(LENET_PATH, map_location=device))
    print('loaded', LENET_PATH)
else:
    opt_l = torch.optim.Adam(lenet.parameters(), lr=3e-4)
    bs_l, t0 = 256, time.time()
    for ep in range(B['lenet_epochs']):
        perm = torch.randperm(x_train.size(0))
        corr = tot = 0
        for i in range(0, x_train.size(0) - bs_l + 1, bs_l):
            j = perm[i:i + bs_l]
            xb, yb = x_train[j].to(device), y_train[j].to(device)
            logit = lenet(xb)
            loss = F.cross_entropy(logit, yb)
            opt_l.zero_grad(); loss.backward(); opt_l.step()
            corr += (logit.argmax(1) == yb).sum().item(); tot += yb.numel()
        print(f'  LeNet epoch {ep+1}/{B["lenet_epochs"]}  train acc {100*corr/tot:.1f}%  [{time.time()-t0:.0f}s]')
    torch.save(lenet.state_dict(), LENET_PATH)

for p in lenet.parameters():
    p.requires_grad_(False)
lenet.eval()

with torch.no_grad():
    acc = sum((lenet(x_test[i:i+1000].to(device)).argmax(1) == y_test[i:i+1000].to(device)).sum().item()
              for i in range(0, 10000, 1000)) / 10000
print(f'frozen LeNet test accuracy: {100*acc:.2f}%')

@torch.no_grad()
def kid_score(gen, n=2000, bs=250):
    """Kernel distance between real and generated images in frozen-LeNet feature space.

    Computed on features the critic never sees, so -- unlike an MMD read off the critic's
    own codes -- it cannot fall while the samples get worse. RQ mixture, unbiased.
    """
    gen.eval()
    rf, ff = [], []
    for i in range(0, n, bs):
        rf.append(lenet.embed(x_test[i:i + bs].to(device)))
        ff.append(lenet.embed(gen(torch.randn(bs, Z_DIM, device=device))))
    fr, fg = torch.cat(rf)[:n], torch.cat(ff)[:n]
    mix = lambda a, b: sum(rq(a, b, alpha=al, ell=1.0) for al in ALPHAS)
    return (_mean_offdiag(mix(fr, fr)) + _mean_offdiag(mix(fg, fg)) - 2.0 * mix(fr, fg).mean()).item()

In [ ]:
# ── The MMD-GAN training loop (shared by Part 2 and Part 3) ──────────────────
from copy import deepcopy

def train_mmd_gan(code_dim, n_iters, n_critic=None, batch=None, ndf=None, ngf=None,
                  lambda_ae=8.0, lr=1e-4, betas=(0.5, 0.9), seed=0,
                  log_every=None, kid_every=None, snap_every=None, tag=''):
    n_critic = n_critic or B['n_critic']; batch = batch or B['batch']
    ndf = ndf or B['ndf']; ngf = ngf or B['ngf']
    log_every = log_every or max(1, n_iters // 10)
    kid_every = kid_every or max(1, n_iters // 5)
    snap_every = snap_every or max(1, n_iters // 4)

    torch.manual_seed(seed)
    G = DCGenerator(Z_DIM, ngf).to(device)
    D = AECritic(1, ndf, ngf, code_dim).to(device)
    G_ema = deepcopy(G).eval()
    opt_G = torch.optim.Adam(G.parameters(), lr=lr, betas=betas)
    opt_D = torch.optim.Adam(D.parameters(), lr=lr, betas=betas)
    it_data = batches(x_train, bs=batch, seed=seed)

    hist = {'iter': [], 'mmd2': [], 'recon': [], 'code_norm': [],
            'kid_iter': [], 'kid': [], 'snaps': []}
    z_fix = torch.randn(64, Z_DIM, device=device)
    t0 = time.time()

    for it in range(1, n_iters + 1):
        # critic: push the codes apart, keep reconstructing
        for _ in range(n_critic):
            real, _ = next(it_data)
            with torch.no_grad():
                fake = G(torch.randn(real.size(0), Z_DIM, device=device))
            code_r, rec_r = D(real)
            code_f, rec_f = D(fake)
            m2 = torch.relu(mmd2_codes(code_r, code_f))
            recon = F.mse_loss(rec_r, real) + F.mse_loss(rec_f, fake)
            loss_D = -torch.sqrt(m2 + EPS) + lambda_ae * recon
            opt_D.zero_grad(set_to_none=True); loss_D.backward(); opt_D.step()

        # generator: pull them back together
        real, _ = next(it_data)
        fake = G(torch.randn(batch, Z_DIM, device=device))
        with torch.no_grad():
            code_r = D.encode(real)
        loss_G = mmd_codes(code_r, D.encode(fake))
        opt_G.zero_grad(set_to_none=True); loss_G.backward(); opt_G.step()

        with torch.no_grad():                        # EMA generator, used for all sampling
            dec = min(0.999, (1.0 + it) / (10.0 + it))   # warm-up: no init bias at short budgets
            for pe, p in zip(G_ema.parameters(), G.parameters()):
                pe.data.mul_(dec).add_(p.data, alpha=1.0 - dec)
            for be, b_ in zip(G_ema.buffers(), G.buffers()):
                be.data.copy_(b_.data)

        if it % log_every == 0 or it == 1:
            hist['iter'].append(it)
            hist['mmd2'].append(m2.item())
            hist['recon'].append(recon.item())
            hist['code_norm'].append(code_f.detach().norm(dim=1).mean().item())
        if it % kid_every == 0 or it == n_iters:
            hist['kid_iter'].append(it); hist['kid'].append(kid_score(G_ema))
            print(f'  {tag}[{it:5d}/{n_iters}] MMD2={m2.item():.4f}  recon={recon.item():.4f}  '
                  f'|code|={hist["code_norm"][-1]:.2f}  KID={hist["kid"][-1]:.4f}  [{time.time()-t0:.0f}s]')
        if it % snap_every == 0 or it == n_iters:
            with torch.no_grad():
                hist['snaps'].append((it, G_ema(z_fix[:16]).cpu()))

    hist['seconds'] = time.time() - t0
    return {'G': G, 'G_ema': G_ema, 'D': D, 'hist': hist, 'code_dim': code_dim}

@torch.no_grad()
def code_snr(run, n_batches=30, batch=128):
    """Scale-free read-out of the training signal: mean MMD^2 over its own spread.

    MMD^2 is not comparable across code widths (the codes live on different scales), but
    signal / sampling-noise is, and it is what decides whether the generator can descend.
    """
    D, G_ema = run['D'], run['G_ema']
    it_data = batches(x_test, bs=batch, seed=123)
    vals = []
    for _ in range(n_batches):
        real, _ = next(it_data)
        fake = G_ema(torch.randn(batch, Z_DIM, device=device))
        vals.append(mmd2_codes(D.encode(real), D.encode(fake)).item())
    vals = np.array(vals)
    return vals.mean(), vals.std(), vals.mean() / max(vals.std(), 1e-12)

### The generator everything else is built on

One MMD-GAN, trained once, then frozen. Part 3 re-runs this same function to vary the critic width;
here we take the width the slides settle on.

In [ ]:
# ── The generator everything below is built on (frozen from here) ────────────
MAIN_CODE_DIM = 32     # the width Part 3 finds best
print(f'training the Part 2 generator:  code_dim = {MAIN_CODE_DIM}, '
      f'{B["main_iters"]} iterations, preset {PRESET!r}')
main = train_mmd_gan(MAIN_CODE_DIM, B['main_iters'], tag='main ', seed=0)

G_frozen, D_frozen = main['G_ema'].eval(), main['D'].eval()
for p in G_frozen.parameters():
    p.requires_grad_(False)
for p in D_frozen.parameters():
    p.requires_grad_(False)

def phi(x):
    """The frozen critic's code — the feature the GAN's kernel was defined on."""
    return D_frozen.encode(x)

h = main['hist']
fig, axes = plt.subplots(1, 4, figsize=(13.5, 3.2))
axes[0].plot(h['iter'], h['mmd2'], color=C1); axes[0].set_yscale('log')
axes[0].set_title(r'$\widehat{\mathrm{MMD}^2}$ on codes'); axes[0].set_xlabel('iteration')
axes[1].plot(h['iter'], h['recon'], color=C1)
axes[1].set_title('critic reconstruction'); axes[1].set_xlabel('iteration')
axes[2].plot(h['iter'], h['code_norm'], color=C1)
axes[2].set_title(r'mean $\|$code$\|$  (spectral norm keeps it bounded)'); axes[2].set_xlabel('iteration')
axes[3].plot(h['kid_iter'], h['kid'], 'o-', color=C1)
axes[3].set_title('KID, frozen LeNet'); axes[3].set_xlabel('iteration')
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, len(h['snaps']) + 1, figsize=(2.4 * (len(h['snaps']) + 1), 2.7))
for ax, (it, imgs) in zip(axes, h['snaps']):
    show_grid(imgs, f'iter {it}', nrow=4, ax=ax)
with torch.no_grad():
    show_grid(G_frozen(torch.randn(64, Z_DIM, device=device)), 'final, $G_{\\rm ema}$', nrow=8, ax=axes[-1])
plt.tight_layout(); plt.show()
print(f'KID: {h["kid"][0]:.4f} (first checkpoint)  ->  {h["kid"][-1]:.4f} (final)   '
      f'[{h["seconds"]:.0f} s]')

### Restoration: gradient descent in $z$,

$$\hat z = \arg\min_z \; f\big(y, G_\theta(z)\big) + \tfrac{\lambda}{2}\|z\|^2$$





In [ ]:
# ── Restoration by gradient descent in z only ───────────────────────────────
NOISE_SIGMA = 0.5

def corrupt(x, sigma=NOISE_SIGMA, seed=0):
    g = torch.Generator(device='cpu').manual_seed(seed)
    noise = torch.randn(x.shape, generator=g).to(x.device)
    return (x + sigma * noise).clamp_(-1.0, 1.0)

def psnr(a, b):
    """Mean PSNR in dB, compared in [0,1]."""
    a01, b01 = (a.clamp(-1, 1) + 1) / 2, (b.clamp(-1, 1) + 1) / 2
    mse = F.mse_loss(a01, b01, reduction='none').flatten(1).mean(1)
    return (10 * torch.log10(1.0 / mse.clamp_min(1e-10))).mean().item()

def rkhs_distance(f_gen, f_tgt):
    """Paired RKHS distance ||phi(G(z)) - phi(y)||^2_H = 2C - 2k(.,.), RBF mixture."""
    f_tgt = f_tgt.detach()
    base = _median_base(sqdist(f_tgt, f_tgt))
    d2 = (f_gen - f_tgt).pow(2).sum(1)
    kpair = sum(torch.exp(-d2 / (c * base)) for c in RBF_MULTS)
    return (2.0 * len(RBF_MULTS) - 2.0 * kpair).clamp_min(0.0).mean()

LAM_DEFAULT = {'pixel': 1.0, 'kernel': 0.05}

def restore_in_z(y_obs, data_term='pixel', n_steps=None, lr=0.05, lam=None,
                 sigma_n=NOISE_SIGMA, restarts=3, seed=0, trace=False):
    r"""argmin_z  f(y, G(z)) + (lam/2)||z||^2,  by Adam on z. G and the critic stay frozen.

    data_term='pixel' : f = ||y - G(z)||^2 / (2 sigma_n^2), the Gaussian-noise negative
        log-likelihood -- so lam = 1 is exactly the MAP estimate under the N(0,I) prior that
        G was trained against, and no weight has to be invented.
    data_term='kernel': f = ||phi(G(z)) - phi(y)||^2_H under the critic's own kernel, the
        distance the GAN was trained with; lam puts the prior on the same scale.
    """
    n_steps = n_steps or B['z_steps']
    lam = LAM_DEFAULT[data_term] if lam is None else lam

    def objective(z, f_tgt):
        x_g = G_frozen(z)
        f = ((x_g - y_obs).pow(2).flatten(1).sum(1).mean() / (2.0 * sigma_n ** 2)
             if data_term == 'pixel' else rkhs_distance(phi(x_g), f_tgt))
        return x_g, f + 0.5 * lam * z.pow(2).sum(1).mean()

    best_x, best_obj, best_hist = None, None, None
    for r in range(restarts):
        torch.manual_seed(seed + r)
        z = (0.1 * torch.randn(y_obs.size(0), Z_DIM, device=device)).requires_grad_(True)
        opt = torch.optim.Adam([z], lr=lr)
        f_tgt = phi(y_obs).detach() if data_term == 'kernel' else None
        hist = []
        for _ in range(n_steps):
            _, obj = objective(z, f_tgt)
            opt.zero_grad(); obj.backward(); opt.step()
            if trace:
                hist.append(obj.item())
        with torch.no_grad():
            x_g, obj = objective(z, f_tgt)
        if best_obj is None or obj.item() < best_obj:
            best_obj, best_x, best_hist = obj.item(), x_g.detach(), hist
    return best_x, best_obj, best_hist

# one clean digit of each class
sel, seen = [], set()
for i in range(len(x_test)):
    c = int(y_test[i])
    if c not in seen:
        sel.append(i); seen.add(c)
    if len(seen) == 10:
        break
sel = sorted(sel, key=lambda i: int(y_test[i]))
x_clean = x_test[sel].to(device)
x_noisy = corrupt(x_clean)

t0 = time.time()
x_pix, _, tr_pix = restore_in_z(x_noisy, 'pixel', trace=True)
x_ker, _, tr_ker = restore_in_z(x_noisy, 'kernel', trace=True)
print(f'[{time.time()-t0:.0f} s]  {B["z_steps"]} Adam steps on z, 3 restarts, '
      f'lambda = {LAM_DEFAULT["pixel"]} (pixel, = MAP) / {LAM_DEFAULT["kernel"]} (kernel)')
print(f'PSNR  noisy vs clean               : {psnr(x_noisy, x_clean):5.2f} dB')
print(f'PSNR  restored, pixel data term    : {psnr(x_pix,   x_clean):5.2f} dB')
print(f'PSNR  restored, kernel data term   : {psnr(x_ker,   x_clean):5.2f} dB')

fig = plt.figure(figsize=(11.5, 5.4))
gs = fig.add_gridspec(1, 2, width_ratios=[2.6, 1])
axl = fig.add_subplot(gs[0])
rows = torch.cat([x_clean, x_noisy, x_pix, x_ker])
show_grid(rows, 'clean  /  noisy ($\\sigma$ = 0.5)  /  restored, pixel $f$  /  restored, kernel $f$',
          nrow=10, ax=axl)
axr = fig.add_subplot(gs[1])
axr.plot(tr_pix, color=C1, label=r'pixel  $\|y-G(z)\|^2$')
axr.plot(tr_ker, color=C2, label=r'kernel  $\|\varphi(G(z))-\varphi(y)\|^2_{\mathcal{H}}$')
axr.set_yscale('log'); axr.set_xlabel('gradient step in $z$'); axr.set_ylabel('objective')
axr.set_title('Descent in $z$, nothing else'); axr.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── How far the prior carries: a noise sweep ────────────────────────────────
eval_x = x_test[:64].to(device)
print(f'{"sigma":>6} | {"noisy":>8} | {"pixel f":>8} | {"kernel f":>9}    PSNR dB, 64 test images')
print('-' * 62)
sweep_rows = []
for s in [0.25, 0.5, 0.75, 1.0]:
    xn = corrupt(eval_x, sigma=s, seed=1)
    xp, _, _ = restore_in_z(xn, 'pixel', sigma_n=s, restarts=2)
    xk, _, _ = restore_in_z(xn, 'kernel', sigma_n=s, restarts=2)
    row = (s, psnr(xn, eval_x), psnr(xp, eval_x), psnr(xk, eval_x))
    sweep_rows.append(row)
    print(f'{row[0]:6.2f} | {row[1]:8.2f} | {row[2]:8.2f} | {row[3]:9.2f}')

sweep_rows = np.array(sweep_rows)
fig, ax = plt.subplots(figsize=(5.0, 3.4))
ax.plot(sweep_rows[:, 0], sweep_rows[:, 1], 'o--', color=INK2, label='noisy input')
ax.plot(sweep_rows[:, 0], sweep_rows[:, 2], 'o-', color=C1, label=r'pixel $f$')
ax.plot(sweep_rows[:, 0], sweep_rows[:, 3], 'o-', color=C2, label=r'kernel $f$')
ax.set_xlabel(r'corruption $\sigma$'); ax.set_ylabel('PSNR (dB)')
ax.set_title('Denoising with no denoiser — only a frozen $G_\\theta$')
ax.legend()
plt.tight_layout(); plt.show()

### An encoder on top of the frozen generator

The encoder maps $(x, c) \mapsto (\mu_\phi, \log \sigma^2_\phi)$ in $z$-space, $z$ is drawn by
reparameterisation, and the **frozen** $G_\theta$ plays the decoder:

$$\mathcal L(\phi) = \underbrace{\big\|G_\theta(z) - x\big\|^2}_{\text{reconstruction}}
\;+\; \beta\, \underbrace{\mathrm{KL}\big(\mathcal N(\mu_\phi, \sigma^2_\phi)\,\|\,\mathcal N(0,I)\big)}_{\text{stay on the prior } G_\theta \text{ knows}}
\;+\; \gamma\, \underbrace{\mathrm{CE}\big(\mathrm{LeNet}(G_\theta(z)),\, c\big)}_{\text{the class signal}}$$



In [ ]:
# ── An encoder on top of the frozen generator ───────────────────────────────
class VAEEncoder(nn.Module):
    """(image, class) -> (mu, log var) in z-space. The generator stays frozen behind it."""
    def __init__(self, z_dim=Z_DIM, ndf=64, n_classes=10, label_dim=16):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, ndf, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(ndf * 2, ndf * 4, 3, 1, 1), nn.LeakyReLU(0.2, True), nn.Flatten())
        self.label_emb = nn.Embedding(n_classes, label_dim)
        self.shared = nn.Sequential(nn.Linear(ndf * 4 * 7 * 7 + label_dim, 512), nn.LeakyReLU(0.2, True))
        self.fc_mu, self.fc_lv = nn.Linear(512, z_dim), nn.Linear(512, z_dim)

    def forward(self, x, labels):
        h = torch.cat([self.cnn(x), self.label_emb(labels)], 1)
        h = self.shared(h)
        return self.fc_mu(h), self.fc_lv(h)

    @staticmethod
    def reparameterise(mu, log_var):
        return mu + torch.exp(0.5 * log_var) * torch.randn_like(mu)

BETA, GAMMA = 0.5, 2.0      # KL weight, class weight
encoder = VAEEncoder().to(device)
opt_e = torch.optim.Adam(encoder.parameters(), lr=3e-4, betas=(0.5, 0.999))
sched_e = torch.optim.lr_scheduler.CosineAnnealingLR(opt_e, T_max=B['enc_iters'])
it_cond = batches(x_train, y_train, bs=B['batch'], seed=11)

hist_e = {'iter': [], 'recon': [], 'kl': [], 'cls': [], 'acc': []}
t0 = time.time()
log_e = max(1, B['enc_iters'] // 10)
for it in range(1, B['enc_iters'] + 1):
    encoder.train()
    xb, yb = next(it_cond)
    mu, log_var = encoder(xb, yb)
    x_hat = G_frozen(encoder.reparameterise(mu, log_var))     # G frozen: gradient flows via z
    l_rec = F.mse_loss(x_hat, xb)
    l_kl = -0.5 * torch.mean(1.0 + log_var - mu.pow(2) - log_var.exp())
    l_cls = F.cross_entropy(lenet(x_hat), yb)                 # LeNet frozen: the class signal
    loss = l_rec + BETA * l_kl + GAMMA * l_cls
    opt_e.zero_grad(set_to_none=True); loss.backward()
    nn.utils.clip_grad_norm_(encoder.parameters(), 5.0)
    opt_e.step(); sched_e.step()
    if it % log_e == 0 or it == 1:
        with torch.no_grad():
            acc = (lenet(x_hat).argmax(1) == yb).float().mean().item()
        for k, v in zip(['iter', 'recon', 'kl', 'cls', 'acc'],
                        [it, l_rec.item(), l_kl.item(), l_cls.item(), acc]):
            hist_e[k].append(v)
        print(f'  enc [{it:5d}/{B["enc_iters"]}] recon={l_rec.item():.4f}  KL={l_kl.item():.4f}  '
              f'cls={l_cls.item():.4f}  acc={100*acc:.1f}%  [{time.time()-t0:.0f}s]')
encoder.eval()

fig, axes = plt.subplots(1, 4, figsize=(13.5, 3.2))
for ax, key, name in zip(axes, ['recon', 'kl', 'cls'],
                         ['reconstruction', 'KL to the prior', 'class cross-entropy']):
    ax.plot(hist_e['iter'], hist_e[key], color=C1); ax.set_title(name); ax.set_xlabel('iteration')
axes[3].plot(hist_e['iter'], [100 * a for a in hist_e['acc']], color=C1)
axes[3].axhline(10, color=INK2, lw=1.0, ls='--')
axes[3].text(hist_e['iter'][0], 13, 'chance', fontsize=8, color=INK2)
axes[3].set_title(r'LeNet accuracy on $G(z)$  (%)'); axes[3].set_xlabel('iteration')
plt.tight_layout(); plt.show()

In [ ]:
# ── Conditional generation, for free ────────────────────────────────────────
by_class = {c: (y_test == c).nonzero().flatten()[:10] for c in range(10)}
rows, accs = [], []
with torch.no_grad():
    for c in range(10):
        xc = x_test[by_class[c]].to(device)
        lc = torch.full((xc.size(0),), c, dtype=torch.long, device=device)
        mu, lv = encoder(xc, lc)
        gen = G_frozen(encoder.reparameterise(mu, lv))
        rows.append(gen)
        accs.append((lenet(gen).argmax(1) == lc).float().mean().item())

fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.6))
show_grid(torch.cat(rows), 'conditional samples — one row per requested class, 0 at the top',
          nrow=10, ax=axes[0])
axes[1].barh(np.arange(10), [100 * a for a in accs], color=C1, height=0.65)
axes[1].axvline(10, color=INK2, lw=1.0, ls='--')
axes[1].set_yticks(np.arange(10)); axes[1].set_ylabel('requested class')
axes[1].set_xlabel('% of generated images the frozen LeNet reads as that class')
axes[1].set_title(f'Class coherence   (mean {100*np.mean(accs):.1f}%)')
axes[1].invert_yaxis(); axes[1].grid(axis='y', visible=False)
plt.tight_layout(); plt.show()

### Straight lines in $Z$

$$z(\alpha) = (1-\alpha)\,\mu_\phi(x_a, c_a) + \alpha\,\mu_\phi(x_b, c_b), \qquad \alpha \in [0,1]$$

If the latent space were merely memorising outputs, a straight line between two codes would pass
through nothing in particular. A smooth morph that stays on the digit manifold is the evidence that
it is organised.

In [ ]:
# ── Straight lines in Z ────────────────────────────────────────────────────
n_steps_i = 10
pairs = [(0, 1), (3, 8), (4, 9), (5, 6), (2, 7)]
strips = []
with torch.no_grad():
    for ca, cb in pairs:
        xa, xb_ = x_test[by_class[ca][:1]].to(device), x_test[by_class[cb][:1]].to(device)
        mu_a, _ = encoder(xa, torch.tensor([ca], device=device))
        mu_b, _ = encoder(xb_, torch.tensor([cb], device=device))
        strips.append(torch.cat([G_frozen((1 - a) * mu_a + a * mu_b)
                                 for a in torch.linspace(0, 1, n_steps_i)]))
show_grid(torch.cat(strips), 'linear interpolation in $Z$ between two encoded digits '
                             '(rows: ' + ', '.join(f'{a}$\\to${b}' for a, b in pairs) + ')',
          nrow=n_steps_i, figsize=(9.5, 5.2))
plt.show()

---
# High-dimensional feature spaces and the vanishing MMD
### *Richer features but lower signal*



On the Gaussian shift problem $P = \mathcal N(0, I_d)$, $Q = \mathcal N(\mu e_1, I_d)$, with
$t = 1/(2\sigma^2)$, the squared MMD is available exactly:

$$\mathrm{MMD}^2(P,Q) \;=\; 2\,(1+4t)^{-d/2}\left[1 - \exp\!\Big(\!-\frac{t\mu^2}{1+4t}\Big)\right]
\;\xrightarrow[d\to\infty]{}\; 0 .$$



In [ ]:
# ── The closed form, against Monte-Carlo ─────────────────────────────────────
def mmd2_gauss_shift(d, mu, sigma):
    """Exact MMD^2 between N(0,I_d) and N(mu e_1, I_d) under the RBF kernel."""
    t = 1.0 / (2.0 * sigma ** 2)
    d = np.asarray(d, dtype=float)
    return 2.0 * (1.0 + 4.0 * t) ** (-d / 2.0) * (1.0 - np.exp(-t * mu ** 2 / (1.0 + 4.0 * t)))

MU_G, SIGMA_G = 2.0, 1.0
d_grid = np.array([1, 2, 4, 8, 16, 24, 32, 48, 64, 96, 128, 192, 256])
n_g, reps_g = 512, 24

def mmd2_median(X, Y):
    """Unbiased MMD^2 with the bandwidth set by the median heuristic (2 sigma^2 = median d^2).

    This is the rule the reference implementation uses, and it is NOT the setting the closed
    form above describes -- the bandwidth grows with d instead of staying put, so it is worth
    carrying both curves.
    """
    pooled = torch.cat([X, Y])
    base = _median_base(sqdist(pooled, pooled))
    k = lambda a, b: torch.exp(-sqdist(a, b) / base)
    return _mean_offdiag(k(X, X)) + _mean_offdiag(k(Y, Y)) - 2.0 * k(X, Y).mean()

torch.manual_seed(0)
mc_mean, mc_std, md_mean, md_std = [], [], [], []
for dw in d_grid:
    vals, vals_m = [], []
    for _ in range(reps_g):
        Xg = torch.randn(n_g, int(dw), device=device)
        Yg = torch.randn(n_g, int(dw), device=device)
        Yg[:, 0] += MU_G
        vals.append(mmd2_unbiased(rbf, Xg, Yg, sigma=SIGMA_G).item())
        vals_m.append(mmd2_median(Xg, Yg).item())
    mc_mean.append(np.mean(vals)); mc_std.append(np.std(vals))
    md_mean.append(np.mean(vals_m)); md_std.append(np.std(vals_m))
mc_mean, mc_std = np.array(mc_mean), np.array(mc_std)
md_mean, md_std = np.array(md_mean), np.array(md_std)
exact = mmd2_gauss_shift(d_grid, MU_G, SIGMA_G)

print(f'P = N(0, I_d)   Q = N({MU_G} e_1, I_d)   sigma = {SIGMA_G}   '
      f'(the means stay {MU_G} apart at every d)')
print(f'{"d":>5} | {"exact MMD^2":>13} | {"Monte-Carlo":>13} | {"MC std":>10} | {"signal / noise":>14}'
      f'   (fixed sigma)')
print('-' * 70)
for i, dw in enumerate(d_grid):
    if dw in (1, 8, 32, 64, 128, 256):
        print(f'{dw:5d} | {exact[i]:13.3e} | {mc_mean[i]:13.3e} | {mc_std[i]:10.3e} | '
              f'{exact[i]/mc_std[i]:14.2f}')
print(f'\nfixed sigma     : MMD^2 falls {exact[0]/exact[-1]:.3g}x between d = 1 and d = 256, '
      f'with mu held fixed at {MU_G}')
print(f'median heuristic: MMD^2 falls {md_mean[0]/md_mean[-1]:.3g}x over the same range '
      f'(a 1/d rate, not an exponential one)')

In [ ]:
# ── What that costs a two-sample test: power vs d ────────────────────────────
def mmd2_u_from_gram(K, m):
    n = K.size(0) - m
    Kxx, Kyy, Kxy = K[:m, :m], K[m:, m:], K[:m, m:]
    return (_mean_offdiag(Kxx) + _mean_offdiag(Kyy) - 2.0 * Kxy.mean())

def perm_pvalue(K, m, n_perm=100):
    """Permutation test on a pooled Gram matrix (the standard kernel two-sample test)."""
    obs = mmd2_u_from_gram(K, m)
    N = K.size(0)
    ge = 1
    for _ in range(n_perm):
        p = torch.randperm(N, device=K.device)
        ge += int(mmd2_u_from_gram(K[p][:, p], m) >= obs)
    return ge / (n_perm + 1)

torch.manual_seed(0)
n_p, reps_p, n_perm = 192, 20, 100
d_pow = np.array([1, 4, 16, 32, 64, 128, 256])
power, power_med = [], []
t0 = time.time()
for dw in d_pow:
    rej = rej_m = 0
    for _ in range(reps_p):
        Xp = torch.randn(n_p, int(dw), device=device)
        Yp = torch.randn(n_p, int(dw), device=device)
        Yp[:, 0] += MU_G
        pooled = torch.cat([Xp, Yp])
        rej += int(perm_pvalue(rbf(pooled, pooled, sigma=SIGMA_G), n_p, n_perm) <= 0.05)
        base = _median_base(sqdist(pooled, pooled))
        rej_m += int(perm_pvalue(torch.exp(-sqdist(pooled, pooled) / base), n_p, n_perm) <= 0.05)
    power.append(rej / reps_p); power_med.append(rej_m / reps_p)
print(f'[{time.time()-t0:.0f} s]  permutation test, n = {n_p} per sample, alpha = 0.05, '
      f'{reps_p} repeats, {n_perm} permutations')

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.5))

ax = axes[0]
ax.plot(d_grid, exact, color=C1, label=r'closed form, $\sigma$ fixed')
ax.errorbar(d_grid, mc_mean, yerr=mc_std, fmt='o', color=C2, ms=5, lw=1.5,
            capsize=2, label=f'Monte-Carlo, $\\sigma$ fixed ($n$={n_g})')
ax.errorbar(d_grid, md_mean, yerr=md_std, fmt='s', color=C3, ms=5, lw=1.5,
            capsize=2, label='Monte-Carlo, median heuristic')
ax.set_xscale('log', base=2); ax.set_yscale('log')
ax.set_xlabel('feature-space width $d$'); ax.set_ylabel(r'$\mathrm{MMD}^2$')
ax.set_title(r'$\mu$ fixed at 2, $d$ growing'); ax.legend(loc='lower left')

ax = axes[1]
ax.plot(d_grid, exact / mc_std, color=C1, label=r'$\sigma$ fixed')
ax.plot(d_grid, md_mean / md_std, color=C3, label='median heuristic')
ax.axhline(1.0, color=INK2, lw=1.0, ls='--')
ax.text(1.2, 1.25, 'signal = sampling noise', fontsize=8, color=INK2)
ax.set_xscale('log', base=2); ax.set_yscale('log')
ax.set_xlabel('feature-space width $d$'); ax.set_ylabel(r'signal / $\mathrm{sd}[\widehat{\mathrm{MMD}^2}]$')
ax.set_title('The signal sinks toward the noise'); ax.legend()

ax = axes[2]
ax.plot(d_pow, power, 'o-', color=C1, label=r'$\sigma$ fixed')
ax.plot(d_pow, power_med, 's-', color=C3, label='median heuristic')
ax.axhline(0.05, color=INK2, lw=1.0, ls='--')
ax.text(1.2, 0.09, r'$\alpha = 0.05$: pure chance', fontsize=8, color=INK2)
ax.set_xscale('log', base=2); ax.set_ylim(-0.05, 1.05)
ax.set_xlabel('feature-space width $d$'); ax.set_ylabel('rejection rate')
ax.set_title('A kernel test stops seeing the difference'); ax.legend()

plt.tight_layout(); plt.show()

### The ablation

> **Experiments on CIFAR-10.** $d = 256$: MMD nearly constant, training collapses to artefacts.
> $d = 32$: best structure. $d = 16$: coherent. We monitored a 3-D PCA of the critic's features to
> see whether the critic was ahead of the generator.

The same ablation on MNIST, so that it runs here: the autoencoder critic of Part 2 at
$d \in \{16, 32, 256\}$, everything else held fixed. $\widehat{\mathrm{MMD}^2}$ is *not* comparable
across widths — the codes live on different scales — so the comparison is made on
signal / sampling-noise, which is scale-free, and on the frozen-LeNet KID, which the critic cannot
game.

In [ ]:
# ── The ablation: d = 16, 32, 256, everything else identical ─────────────────
CODE_DIMS = [16, 32, 256]
t0 = time.time()
abl = {}
for cd in CODE_DIMS:
    print(f'code_dim = {cd}')
    abl[cd] = train_mmd_gan(cd, B['abl_iters'], tag=f'd={cd} ', seed=0)
print(f'[total {time.time()-t0:.0f} s,  {B["abl_iters"]} generator iterations each, preset {PRESET!r}]')

print()
print(f'{"code dim":>9} | {"MMD^2 on codes":>15} | {"its spread":>11} | {"signal/noise":>12} | {"KID (frozen LeNet)":>19}')
print('-' * 78)
snrs = {}
for cd in CODE_DIMS:
    mu_, sd_, s_ = code_snr(abl[cd])
    snrs[cd] = (mu_, sd_, s_)
    print(f'{cd:9d} | {mu_:15.5f} | {sd_:11.5f} | {s_:12.2f} | {abl[cd]["hist"]["kid"][-1]:19.4f}')

### The 3-D PCA monitor

The slide's diagnostic: project the critic's codes for real and generated batches onto their first
three principal components. If the clouds are cleanly separated the critic is ahead of a generator
that is receiving nothing it can act on; if they mix, the generator has caught up. The logistic probe
puts a number on the same question.

In [ ]:
# ── The 3-D PCA monitor of the slide: is the critic ahead of the generator? ──
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

fig = plt.figure(figsize=(12.5, 3.8))
print(f'{"code dim":>9} | {"probe AUC on the codes":>23} | reading')
print('-' * 62)
for j, cd in enumerate(CODE_DIMS):
    run = abl[cd]
    with torch.no_grad():
        real, _ = next(batches(x_test, bs=512, seed=7))
        fake = run['G_ema'](torch.randn(512, Z_DIM, device=device))
        cr, cf = run['D'].encode(real), run['D'].encode(fake)

    codes = torch.cat([cr, cf]).cpu().numpy()
    lab = np.r_[np.zeros(512), np.ones(512)]
    half = np.r_[np.arange(0, 256), np.arange(512, 768)]
    rest = np.r_[np.arange(256, 512), np.arange(768, 1024)]
    probe = LogisticRegression(max_iter=2000).fit(codes[half], lab[half])
    auc = roc_auc_score(lab[rest], probe.decision_function(codes[rest]))
    verdict = ('critic far ahead — the codes are trivially separable' if auc > 0.95 else
               'critic ahead' if auc > 0.75 else 'generator has caught up — real and fake codes mix')
    print(f'{cd:9d} | {auc:23.3f} | {verdict}')

    Xc = torch.cat([cr, cf]).cpu()
    Xc = Xc - Xc.mean(0, keepdim=True)
    U, S, V = torch.pca_lowrank(Xc, q=3)
    P3 = (Xc @ V[:, :3]).numpy()
    ax = fig.add_subplot(1, 3, j + 1, projection='3d')
    ax.scatter(*P3[:512].T, s=5, color='#104281', alpha=0.55, label='real')
    ax.scatter(*P3[512:].T, s=5, color=C2, alpha=0.55, label='generated')
    ax.set_title(f'$d$ = {cd}   (AUC {auc:.2f})')
    ax.set_xticklabels([]); ax.set_yticklabels([]); ax.set_zticklabels([])
    ax.grid(False)
    if j == 0:
        ax.legend(loc='upper left')
plt.suptitle("3-D PCA of the critic's codes — the slide's monitor", y=1.03)
plt.tight_layout(); plt.show()